### Real-Time Data Processing with Apache Spark

#### Introduction
In this session, we explore the use of Apache Spark to handle real-time event data coming from our music streaming service, which is streamed into Kafka. We will leverage this real-time data, combine it with static data from our SQL database, and demonstrate how to perform joins between live streaming data and historical records. This setup is crucial for understanding the dynamics of our streaming service usage and for making data-driven decisions in real-time.

1. **Environment Setup**: Configure the environment by specifying paths and settings for PostgreSQL JDBC and Kafka connections. This setup includes the necessary drivers and connection details to access our PostgreSQL database and Kafka topics.

In [2]:
import os

# Path to the PostgreSQL JDBC jar file
rel_path = os.path.relpath('./libs/postgresql-42.7.3.jar')

# Connection URL and properties for PostgreSQL
url = "jdbc:postgresql://fhtw-big-data.postgres.database.azure.com/music_store"
postgres_options = {
    "url": url,
    "user": "student",
    "password": "reRZ2pjg1WxqlwjU",
    "driver": "org.postgresql.Driver"
}

2. **Configure Kafka:** We are now reading messages streamed to the "music_streams" kafka topic

In [3]:
# Kafka configuration parameters for Confluent Cloud
kafka_params = {
    "kafka.bootstrap.servers": "46.225.20.89:9092",
    "subscribe": "music-fhtw",
    "kafka.security.protocol": "PLAINTEXT",
    "startingOffsets": "latest",
}

3. **Spark with Kafka + Postgres**: As usual configuring our SparkSession to include kafka + sql

In [4]:
import os
from pyspark.sql import SparkSession

# Create a Spark session and configure it to use the JDBC jar
spark = SparkSession.builder \
    .appName("PostgreSQL Integration Example") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.4.0") \
    .config("spark.jars", rel_path) \
    .config("spark.executor.extraClassPath", rel_path) \
    .config("spark.driver.extraClassPath", rel_path) \
    .getOrCreate()

26/06/08 18:16:17 WARN Utils: Your hostname, codespaces-f5202d resolves to a loopback address: 127.0.0.1; using 10.0.0.231 instead (on interface eth0)
26/06/08 18:16:17 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/usr/local/sdkman/candidates/spark/3.5.1/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/vscode/.ivy2/cache
The jars for the packages stored in: /home/vscode/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-294b41b5-690e-4517-9888-4e90658612e5;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.4.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.4.0 in central
	found org.apache.kafka#kafka-clients;3.3.2 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.9.1 in central
	found org.slf4j#slf4j-api;2.0.6 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central
:: resolution report :: resolve 793ms :: artifacts dl 27ms
	:: 

4. **Inspect incoming data:** First let's have a look at our incoming data from the data stream

In [5]:
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType

# Read streaming data from Kafka
df = spark \
    .readStream \
    .format("kafka") \
    .options(**kafka_params) \
    .load()

# Select and cast the key and value columns to STRING
df_message = df.selectExpr("CAST(key AS STRING)", "CAST(value AS STRING)")

In [6]:
# truncate false
query = df_message.writeStream \
    .outputMode("append") \
    .format("console") \
    .option("truncate", "false") \
    .start()

import time
time.sleep(30)
query.stop()

26/06/08 18:17:54 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-00f0954e-984d-42bd-a896-0d2158b2f13f. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/06/08 18:17:54 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/06/08 18:17:55 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


-------------------------------------------
Batch: 0
-------------------------------------------
+---+-----+
|key|value|
+---+-----+
+---+-----+



-------------------------------------------
Batch: 1
-------------------------------------------


+---+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|key|value                                                                                                                                                                                                                                                                                                                                                                                                                             

-------------------------------------------
Batch: 3
-------------------------------------------
+----+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|key |value                                                                                                                                                                                                                                                                                                                                                                                 

-------------------------------------------
Batch: 5
-------------------------------------------
+----+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|key |value                                                                                                                                                                                                                                                                                                                                                                                       

-------------------------------------------
Batch: 6
-------------------------------------------
+---+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|key|value                                                                                                                                                                                                                                                                                                                            

-------------------------------------------
Batch: 8
-------------------------------------------
+----+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|key |value                                                                                                                                                                                                                                                                                                                                                                                              

-------------------------------------------
Batch: 9
-------------------------------------------
+---+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|key|value                                                                                                                                                                                                                                                                                                                            

### 2. Defining the Kafka Data Schema
To accurately process the streaming data coming from Kafka, we need to define a schema that matches the structure of the JSON data being streamed. This schema setup ensures that Spark can correctly parse and manipulate the data fields as required for our processing logic.

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, LongType, IntegerType

# Define the schema for the JSON data from Kafka
json_schema = StructType([
    StructField("ts", LongType(), True),
    StructField("auth", StringType(), True),
    StructField("page", StringType(), True),
    StructField("song", StringType(), True),
    StructField("level", StringType(), True),
    StructField("artist", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("method", StringType(), True),
    StructField("status", IntegerType(), True),
    StructField("userId", StringType(), True),
    StructField("lastName", StringType(), True),
    StructField("location", StringType(), True),
    StructField("track_id", IntegerType(), True),
    StructField("firstName", StringType(), True),
    StructField("sessionId", IntegerType(), True),
    StructField("userAgent", StringType(), True),
    StructField("registration", LongType(), True),
    StructField("itemInSession", IntegerType(), True)
])

### Let's parse it and display it 

In [ ]:
# Parse the JSON data and apply schema
df_value = df_message.selectExpr("CAST(value AS STRING) as json_string")
parsed_df = df_value.withColumn("jsonData", from_json(col("json_string"), json_schema)).select("jsonData.*")

In [ ]:
# truncate false
query = parsed_df.writeStream \
    .outputMode("append") \
    .format("console") \
    .option("truncate", "false") \
    .start()

import time
time.sleep(30)
query.stop()

#### Aggregating and Displaying Stream Data
Once the streaming data has been parsed and structured, we can perform meaningful aggregations. Here, we aggregate the data by the `level` field, which might represent different user subscription levels in our music streaming service.​

In [ ]:
from pyspark.sql.functions import col

levels_counts = (parsed_df
                 .groupBy("level")
                 .count())

# truncate false
query = levels_counts.writeStream \
    .outputMode("complete") \
    .format("console") \
    .option("truncate", "false") \
    .start()

import time
time.sleep(90)
query.stop()

After obtaining real-time event data from Kafka, our next step is to enrich this stream by joining it with static data that categorizes music tracks by genre. This static data is stored in a PostgreSQL database and includes genre classifications for each track. By joining the streaming data with this genre information, we can perform more detailed analysis and aggregation based on genres, which can be crucial for understanding music preferences and streaming behaviors.

### Joining Kafka Stream with Data from SQL and Aggregating by Genre

First, load your SQL data into a DataFrame, and then join it with the stream. For the join to work efficiently in a streaming context, consider that the join keys must be present in both datasets, and watermarks might be necessary to manage out-of-order data if dealing with windowed aggregations or if your stream is append-only.

In [ ]:
df_genres = spark.read.jdbc(url=url, 
                            table="""(
                            SELECT tracks.id AS track_id, g.name AS genre 
                            FROM tracks 
                            JOIN public.genres g ON g.id = tracks.genre_id) 
                            AS genre_data
                            """, 
                            properties=postgres_options)

In [ ]:
df_genres.show(10)

### Join with Kafka Stream and Aggregate by Genre.

The join is based on the `track_id`, ensuring that each record in the streaming data is enriched with the corresponding genre information from the static `df_genres` DataFrame. 

This allows us to analyze streaming events in the context of music genres, adding depth to our real-time analytics.

In [ ]:
# Join the data 
df_joined = parsed_df.join(df_genres, parsed_df.track_id == df_genres.track_id, "inner")

In [ ]:
query = df_joined.writeStream \
    .outputMode("append") \
    .format("console") \
    .option("truncate", "false") \
    .start()

import time
time.sleep(90)
query.stop()

### Aggregating Streaming Data by Genre

After joining the real-time streaming data with static genre information, we can further analyze the data by aggregating it based on music genres. This aggregation is crucial for understanding the distribution and popularity of different genres among listeners in real-time.

In [ ]:
# remember df_joined = parsed_df.join(df_genres, parsed_df.track_id == df_genres.track_id, "inner")
genre_counts = df_joined.groupBy("genre").count()

In [ ]:
spark.conf.set("spark.sql.shuffle.partitions", "4")

In [ ]:
query = genre_counts \
    .writeStream \
    .outputMode("complete") \
    .format("console") \
    .option("truncate", "false") \
    .start()

import time
time.sleep(90)
query.stop()